In [0]:
bronze = spark.table("workspace.default.bronze_orders")
print("rows:", bronze.count())
print("distinct order_ids:", bronze.select("order_id").distinct().count())

rows: 60600
distinct order_ids: 60000


In [0]:
from pyspark.sql import functions as F

silver = (
    bronze
    .dropDuplicates(["order_id"])
    .withColumn("customer_name", F.initcap(F.lower(F.trim("customer_name"))))
    .withColumn("region", F.coalesce(F.col("region"), F.lit("Unknown")))
    .withColumn("discount", F.coalesce(F.col("discount"), F.lit(0.0)))
    .withColumn("is_return", F.col("quantity") < 0)
    .withColumn("quantity", F.abs(F.col("quantity")))
    .withColumn("line_revenue",
        F.round(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount")), 2))
)

silver.write.mode("overwrite").format("delta").saveAsTable("workspace.default.silver_orders")

print("bronze:", bronze.count(), "silver:", silver.count())

bronze: 60600 silver: 60000


In [0]:
base = "workspace.default"

# 1. Monthly revenue trend
spark.sql(f"""
CREATE OR REPLACE TABLE {base}.gold_monthly_revenue AS
SELECT date_trunc('month', order_date) AS month,
       COUNT(DISTINCT order_id)        AS orders,
       ROUND(SUM(line_revenue), 2)     AS revenue,
       ROUND(SUM(line_revenue) / COUNT(DISTINCT order_id), 2) AS avg_order_value
FROM {base}.silver_orders
WHERE order_status = 'Completed'
GROUP BY 1
""")

# 2. Revenue by region and category
spark.sql(f"""
CREATE OR REPLACE TABLE {base}.gold_region_category AS
SELECT region, category,
       COUNT(DISTINCT order_id)    AS orders,
       ROUND(SUM(line_revenue), 2) AS revenue
FROM {base}.silver_orders
WHERE order_status = 'Completed'
GROUP BY region, category
""")

# 3. Top customers (ranked with a window function)
spark.sql(f"""
CREATE OR REPLACE TABLE {base}.gold_top_customers AS
SELECT customer_id, customer_name, segment, revenue,
       RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
FROM (
  SELECT customer_id, customer_name, segment,
         ROUND(SUM(line_revenue), 2) AS revenue
  FROM {base}.silver_orders
  WHERE order_status = 'Completed'
  GROUP BY customer_id, customer_name, segment
)
""")

# 4. Order status mix (completed vs returned vs cancelled)
spark.sql(f"""
CREATE OR REPLACE TABLE {base}.gold_status_summary AS
SELECT order_status,
       COUNT(*) AS orders,
       ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_orders
FROM {base}.silver_orders
GROUP BY order_status
""")

# Quick check
spark.sql(f"SELECT * FROM {base}.gold_monthly_revenue ORDER BY month").show(30)
spark.sql(f"SELECT * FROM {base}.gold_status_summary").show()

+-------------------+------+----------+---------------+
|              month|orders|   revenue|avg_order_value|
+-------------------+------+----------+---------------+
|2024-01-01 00:00:00|  2343|3019115.18|        1288.57|
|2024-02-01 00:00:00|  2111|2758666.17|        1306.81|
|2024-03-01 00:00:00|  2331|2973974.96|        1275.84|
|2024-04-01 00:00:00|  2268| 2907756.3|        1282.08|
|2024-05-01 00:00:00|  2292| 2958377.2|        1290.74|
|2024-06-01 00:00:00|  2208|2855677.91|        1293.33|
|2024-07-01 00:00:00|  2223|2815292.02|        1266.44|
|2024-08-01 00:00:00|  2242|3049662.51|        1360.24|
|2024-09-01 00:00:00|  2263|2941406.42|        1299.78|
|2024-10-01 00:00:00|  2359|3103184.62|        1315.47|
|2024-11-01 00:00:00|  2198|2800412.98|        1274.07|
|2024-12-01 00:00:00|  2394|3061788.21|        1278.94|
|2025-01-01 00:00:00|  2294|2957935.78|        1289.42|
|2025-02-01 00:00:00|  2076|2699647.82|        1300.41|
|2025-03-01 00:00:00|  2308|2958415.19|        1

In [0]:
silver_total = spark.sql("""
  SELECT ROUND(SUM(line_revenue), 2) AS t FROM workspace.default.silver_orders
  WHERE order_status = 'Completed'""").first()["t"]
gold_total = spark.sql("""
  SELECT ROUND(SUM(revenue), 2) AS t FROM workspace.default.gold_monthly_revenue""").first()["t"]
region_total = spark.sql("""
  SELECT ROUND(SUM(revenue), 2) AS t FROM workspace.default.gold_region_category""").first()["t"]

print("silver:", silver_total, "| gold monthly:", gold_total, "| gold region:", region_total)
print("MATCH" if silver_total == gold_total == region_total else "MISMATCH")

silver: 69938303.27 | gold monthly: 69938303.27 | gold region: 69938303.27
MATCH
